### Fase 1: Transformando Canciones en "Canastas"
El algoritmo Apriori no lee DataFrames normales; necesita una lista de transacciones. 
En nuestro caso, cada **Canción** es una transacción, y los **Artistas** que participan en ella son los artículos de la canasta.

In [4]:
# Instalar si es necesario: !pip install mlxtend
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. INGESTA DE DATOS
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"
df_spotify = pd.read_csv(url, encoding='latin-1')

# 2. LIMPIEZA: Eliminar nulos de la columna de artistas
df_spotify = df_spotify.dropna(subset=['Artist'])

# 3. TRANSFORMACIÓN: Separar los artistas que colaboran en cada canción
# Asumimos que las colaboraciones están separadas por coma (ej. "Bad Bunny, Feid")
# Convertimos esa columna en una verdadera lista de Python
lista_colaboraciones = df_spotify['Artist'].apply(lambda x: [artista.strip() for artista in x.split(',')]).tolist()

# Muestra rápida de cómo se ven nuestras "canastas"
print("Muestra de las primeras 5 colaboraciones:")
for i in range(5):
    print(f"Canción {i+1}: {lista_colaboraciones[i]}")


Muestra de las primeras 5 colaboraciones:
Canción 1: ['Tommy Richman']
Canción 2: ['Kendrick Lamar']
Canción 3: ['Artemas']
Canción 4: ['Miley Cyrus']
Canción 5: ['Eminem']


In [5]:

# Utiliza TransactionEncoder para convertir 'lista_colaboraciones' en una matriz booleana (One-Hot)
# Guárdala en un DataFrame llamado 'df_transacciones'

te = TransactionEncoder()
matrix_bool = te.fit_transform(lista_colaboraciones)
df_transacciones = pd.DataFrame(matrix_bool, columns=te.columns_)

print("--- MATRIZ TRANSACCIONAL ---")
display(df_transacciones)

--- MATRIZ TRANSACCIONAL ---


,"""XY""",$OHO BANI,$uicideboy$,&ME,(G)I-DLE,*NSYNC,.diedlonely,10CM,13 Organisï¿,1da Banton,...,ýýýýýýýýýý,ýýýýýýýýýý ýýýýýýýý,ýýýýýýýýýýýý,ýýýýýýýýýýýý ýýýýýýýýýýýýýý,ýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýý(ýýýýýýýýýýýýýýý),ýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýýýýý
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4590,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4591,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4592,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4593,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### Fase 2: Aplicando Apriori
Ahora que tenemos la matriz con ceros y unos, extraigamos los conjuntos de artistas que colaboran frecuentemente.
Debido a que hay miles de artistas solistas, las colaboraciones son estadísticamente raras. Usaremos un `min_support` muy bajo para capturar las colaboraciones.

In [6]:
# --- RETO PARA EL ALUMNO AQUÍ ---

# 1. Encuentra los Itemsets Frecuentes usando un soporte mínimo del 0.005 (0.5% de las canciones)
itemsets_frecuentes = apriori(df_transacciones, min_support=0.005, use_colnames=True)

# 2. Genera las Reglas de Asociación filtrando por la métrica 'lift' con un mínimo de 1.2
reglas_musicales = association_rules(itemsets_frecuentes, metric="lift", min_threshold=1.2)

# 3. Muestra las reglas resultantes ordenadas por 'confidence' de mayor a menor
print("--- REGLAS DE ASOCIACIÓN BASE (min_support = 0.005 en df_transacciones) ---")
display(reglas_musicales.sort_values(by="confidence", ascending=False))

#VVV Esta parte esta hecha con IA porque al momento de correr las lineas que estan arriba ^^^, no salia absolutamente nada y la neta no supe el porque :)
import re

def extraer_colaboraciones_completas(row):
    artistas = set()
    art_col = str(row['Artist']).strip()
    if art_col == "Tyler, The Creator":
        artistas.add("Tyler, The Creator")
    else:
        for a in art_col.split(','):
            if a.strip():
                artistas.add(a.strip())
    
    track = str(row['Track'])
    match = re.search(r'\((?:feat\.|with)\s+([^)]+)\)', track, re.IGNORECASE)
    if match:
        feats_str = match.group(1)
        partes = re.split(r',|\&|\band\b', feats_str)
        for p in partes:
            p_limpio = p.strip()
            if p_limpio:
                artistas.add(p_limpio)
    return list(artistas)

colabs_reales = df_spotify.apply(extraer_colaboraciones_completas, axis=1).tolist()
te_real = TransactionEncoder()
df_tx_reales = pd.DataFrame(te_real.fit_transform(colabs_reales), columns=te_real.columns_)

# En 4,595 canciones top, un soporte de 0.0005 (aparición conjunta en al menos 3 éxitos mundiales)
# permite extraer las duplas más cotizadas de la industria musical:
itemsets_reales = apriori(df_tx_reales, min_support=0.0005, use_colnames=True)
reglas_musicales_reales = association_rules(itemsets_reales, metric="lift", min_threshold=1.2)

print(f"\n--- REGLAS CON COLABORACIONES REALES (Top Duplas de la Industria, {len(reglas_musicales_reales)} reglas) ---")
display(reglas_musicales_reales.sort_values(by="confidence", ascending=False)[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))


--- REGLAS DE ASOCIACIÓN BASE (min_support = 0.005 en df_transacciones) ---


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski



--- REGLAS CON COLABORACIONES REALES (Top Duplas de la Industria, 30 reglas) ---


,antecedents,consequents,support,confidence,lift
5,(Bryant Myers),(Dei V),0.000653,1.000000,919.000000
14,(PARTYNEXTDOOR),(Drake),0.000653,0.750000,40.544118
4,(Dei V),(Bryant Myers),0.000653,0.600000,919.000000
19,(Sexyy Red),(Drake),0.000653,0.500000,27.029412
7,(DJ Khaled),(Drake),0.000653,0.428571,23.168067
8,(DJ Snake),(Ozuna),0.000653,0.333333,85.092593
24,(Lil Durk),(Lil Baby),0.000653,0.214286,28.132653
2,(Metro Boomin),(21 Savage),0.000653,0.187500,28.718750
22,(Young Thug),(Gunna),0.000653,0.176471,40.544118
26,(Lil Uzi Vert),(Nicki Minaj),0.000653,0.176471,36.858289


### 🧠 Fase 3: Interpretación de Negocio (El Entregable)

Como Científico de Datos, no basta con entregar una tabla. Debes explicarle los hallazgos al Director Creativo. Revisa tu DataFrame `reglas_musicales` y responde las siguientes 4 preguntas:

**1. Análisis de Soporte (El Volumen):**
Menciona a los dos artistas que conforman la regla con el mayor **Soporte**. ¿Qué significa este número en términos simples para el negocio?

> **Tu Respuesta:**
> - Hay cinco parejas empatadas con el soporte más alto: Drake y 21 Savage, Metro Boomin y 21 Savage, Drake y Future, Drake y Lil Baby, y Dei V y Bryant Myers. Todas tienen un soporte de 0.000653 (0.065%), porque cada una colaboró en 3 canciones del top de 2024.
> - En términos de negocio, el soporte dice qué tan seguido aparece algo en el total de datos. Aquí, de las 4,595 canciones del dataset, solo el 0.065% son temas donde participan los dos artistas juntos. Sirve para saber qué tan frecuente y comercialmente relevante es esa dupla.

**2. Análisis de Confianza (La Dirección):**
Busca una regla que tenga una **Confianza** mayor al 0.5 (50%). Traduce esta regla matemática a una frase que el director pueda entender (Ej. "Si contratamos al artista A, existe un X% de probabilidad de que el artista B también deba estar en el track para que sea el éxito que registramos").

> **Tu Respuesta:**
> - La regla es {Bryant Myers} -> {Dei V}, con una confianza de 0.60 (60%). También está {PARTYNEXTDOOR} -> {Drake} con 0.556 (55.6%).
> - Para el director, sería algo así: “Si contratamos a Bryant Myers para un sencillo, en el 60% de los casos en que él aparece en el ranking también aparece Dei V en el mismo track”. Es decir, en los datos históricos, cuando uno está presente, el otro casi siempre lo acompaña.

**3. Análisis de Lift (El Impulso):**
Busca la regla con el **Lift** más alto de toda tu tabla. 
¿Por qué esta métrica es más confiable que el Soporte para decidir si estos dos artistas *realmente* tienen química musical o si solo fue una coincidencia estadística?

> **Tu Respuesta:**
> - La regla con el lift más alto es {Bryant Myers} -> {Dei V} (y su inversa), con un lift de 919. Entre los artistas más mainstream destacan {Ozuna} <-> {DJ Snake} con 85.09, {PARTYNEXTDOOR} -> {Drake} con 40.54 y {Metro Boomin} -> {21 Savage} con 28.72.
> - El soporte solo cuenta cuántas veces aparece algo, así que favorece a los artistas que tienen muchas canciones en el ranking (como Taylor Swift o Bad Bunny) aunque casi nunca colaboren con nadie. El lift, en cambio, compara lo que se observa contra lo que pasaría por puro azar: P(A ∩ B) / [P(A) · P(B)]. Un lift de 28 o de 919 significa que esos dos artistas coinciden muchísimo más de lo que se esperaría si cada uno apareciera por separado, y eso apunta a una colaboración real y a audiencias que se complementan, no a una coincidencia.

**4. Toma de Decisión:**
Basado puramente en las reglas matemáticas que descubriste, si la disquera solo tiene presupuesto para juntar a dos artistas en el estudio la próxima semana, ¿qué pareja elegirías y por qué? (Justifica usando las 3 métricas de oro).

> **Tu Respuesta:**
> - Elegiría a Drake y 21 Savage (Metro Boomin y 21 Savage sería mi segunda opción).
>   1. Soporte: están entre las parejas con mayor soporte conjunto (0.065%), con 3 éxitos juntos en el top de 2024, así que ya se sabe que su música llega a mucha gente.
>   2. Confianza: {21 Savage} -> {Drake} tiene una confianza de 21.4%, o sea que aproximadamente 1 de cada 5 canciones de 21 Savage en el ranking incluye a Drake.
>   3. Lift: es de 5.41, lo que indica que coinciden bastante más de lo que dictaría el azar. En el caso de Metro Boomin y 21 Savage el lift sube a 28.72.
> - Si en cambio la disquera quisiera apostar por el mercado urbano latino emergente, la mejor opción sería Bryant Myers y Dei V, porque tienen la confianza más alta (60%) y el lift más alto de todo el análisis (919). Eso sí, con un soporte tan bajo, esa cifra se basa en muy pocas canciones.